# L29 Lab: Range-Doppler and MTI, notebook

This notebook runs the course's radar chain, `radar_lab.py`, on the bursts you saved with the Phaser app's **Save Capture**. You do not write the chain; you run it, change what each task names, and record what you see in Table F of the L29 Lab packet. There are three tasks, (a) to (c), about 30 to 45 minutes in all, and about five changed lines.

**To run it:** unzip the lab's bundle, `L29_lab.zip`, into your Module 4 folder. It holds this notebook, `radar_lab.py`, the data README, and every recorded burst the cells open (`L29_*.npz`), so the notebook runs as shipped before you change anything. Save your own bursts into the same folder, then run the cells in order from the top.

If your own bursts failed, the recorded bursts work in every cell; say in the packet which rows are recorded rather than your own. Upload the `.ipynb`, with its edits and plots, with the packet.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import radar_lab as rl

# Where the captures are: this folder (the unzipped bundle), or the course's
# layout, notebooks/ beside radar-data/.
DATA = Path(".") if Path("L29_hand_closing.npz").exists() else Path("../radar-data")

# Your hand burst with the wall in view (Procedures B and E). Change the name to your own file.
CAPTURE = DATA / "L29_hand_closing.npz"
cap = rl.load_capture(CAPTURE)
lam = rl.C / cap["fc"]
n_chirps, n_samp = cap["iq"].shape
print(f"{CAPTURE.name}: {n_chirps} chirps of {n_samp} samples, fc = {cap['fc']/1e9:.2f} GHz, "
      f"B = {cap['B']/1e6:.0f} MHz, T_ramp = {cap['T_ramp']*1e6:.0f} us, PRI = {cap['PRI']*1e3:.2f} ms, "
      f"fs = {cap['fs']/1e6:.1f} MSPS, f0 = {cap['f0']/1e3:.1f} kHz")
print(f"cells: {rl.C/(2*cap['B']):.2f} m in range, {lam/(2*n_chirps*cap['PRI']):.3f} m/s in velocity; "
      f"velocity limit +/-{lam/(4*cap['PRI']):.1f} m/s")

In [ ]:
# Readouts used by the tasks. Run this cell; you do not need to change it.
def strongest(rd, r, v, r_lo, r_hi, v_lo, v_hi):
    """Range index, velocity index, and level of the strongest cell in a box of the map."""
    m = ((r >= r_lo) & (r <= r_hi))[:, None] & ((v >= v_lo) & (v <= v_hi))[None, :]
    i, j = np.unravel_index(np.argmax(np.where(m, rd, -400)), rd.shape)
    return i, j, rd[i, j]

def mover(rd, r, v, r_lo=0.3, r_hi=6.0):
    """The strongest cell moving faster than 0.3 m/s either way, between r_lo and r_hi m."""
    return max(strongest(rd, r, v, r_lo, r_hi, 0.3, 15), strongest(rd, r, v, r_lo, r_hi, -15, -0.3),
               key=lambda c: c[2])

def peak_velocity(row, j, v):
    """Velocity of the peak at index j of row, between cells: a parabola through it and its neighbors in dB."""
    a, b, c = row[j - 1], row[j], row[(j + 1) % len(row)]
    return v[j] + 0.5 * (a - c) / (a - 2 * b + c) * (v[1] - v[0])

def floor(rd, r, v, v_lo=-15, v_hi=15):
    """Median level of the map from 6 to 20 m, between v_lo and v_hi m/s: the noise floor there."""
    return np.median(rd[(r > 6) & (r < 20)][:, (v >= v_lo) & (v <= v_hi)])

def width_3db(row, j, step):
    """Width 3 dB down of the peak at index j of row, interpolated between cells.

    A row of the map (rd[i]) gives the width in velocity; a column (rd[:, j]) gives it in range."""
    top = row[j] - 3
    lo = hi = j
    while lo > 0 and row[lo - 1] > top: lo -= 1
    while hi < len(row) - 1 and row[hi + 1] > top: hi += 1
    left = lo - (row[lo] - top) / (row[lo] - row[lo - 1])
    right = hi + (row[hi] - top) / (row[hi] - row[hi + 1])
    return (right - left) * step

def lines(row, v, v_min=0.3, below=10):
    """Velocities of the spectral lines in a row: local peaks within `below` dB of the strongest, |v| > v_min."""
    keep = np.abs(v) > v_min
    top = row[keep].max()
    j = [k for k in range(1, len(row) - 1)
         if keep[k] and row[k] >= row[k - 1] and row[k] >= row[k + 1] and row[k] > top - below]
    return np.array([peak_velocity(row, k, v) for k in j])

def show(rd, r, v, title, r_max=4.0, v_max=5.0, vmin=-120, vmax=-20, ax=None):
    """Draw a range-Doppler map, range up and velocity across, positive closing to the right."""
    ax = ax or plt.figure(figsize=(7, 3.5)).gca()
    keep = r <= r_max
    im = ax.pcolormesh(v, r[keep], rd[keep], vmin=vmin, vmax=vmax, cmap="Blues", shading="auto")
    ax.set_xlim(-v_max, v_max)
    ax.set_xlabel("velocity (m/s), + closing"); ax.set_ylabel("range (m)"); ax.set_title(title)
    plt.colorbar(im, ax=ax, label="dBFS")
    return ax

## Your range-Doppler map

No task here. The cell makes the map of your burst with `rl.range_doppler` and reads four things from it: the strongest thing that moves (faster than 0.3 m/s either way), with its width 3 dB down in range; the strongest thing that does not (0 m/s, beyond 1.5 m), which is the wall in the recorded data; the leakage, which is always the first range cell at 0 m/s; and the noise floor.

**Check:** the recorded hand reads +0.98 m/s at 1.07 m, positive because it closes, 0.42 m wide in range. The wall reads 0 m/s at 2.57 m, −18.6 dBFS, and it is the brightest cell on the map. The leakage sits in the first range cell, 0.17 m (the cell nearest 0 m for this kit's 0 m frequency), at −21.9 dBFS: a little weaker than the wall. The floor is −109.8 dBFS. Your own hand and wall should land where you put them, to within a cell. For Table D's 200 MHz row, set `CAPTURE` to `L29_hand_closing_B200.npz` (the app's presets are all 500 MHz, so that row comes from the recorded burst) and run this cell again: the hand reads 0.97 m wide in range, about 2.5 times wider.

In [ ]:
rd, r, v = rl.range_doppler(cap)
i_hand, j, lvl = mover(rd, r, v)
iw, jw, lw = strongest(rd, r, v, 1.5, 6.0, -0.01, 0.01)
j0 = int(np.argmin(np.abs(v)))
hand_r, hand_v, hand_db = r[i_hand], peak_velocity(rd[i_hand], j, v), lvl
hand_width_m = width_3db(rd[:, j], i_hand, r[1] - r[0])
wall_r, wall_db = r[iw], lw
leak_r, leak_db = r[0], rd[0, j0]
floor_db = floor(rd, r, v)
print(f"strongest mover: {hand_r:.2f} m, {hand_v:+.2f} m/s, {hand_db:.1f} dBFS, {hand_width_m:.2f} m wide in range 3 dB down")
print(f"strongest static return beyond 1.5 m: {wall_r:.2f} m, {wall_db:.1f} dBFS")
print(f"leakage, the first range cell at 0 m/s: {leak_r:.2f} m, {leak_db:.1f} dBFS")
print(f"noise floor: {floor_db:.1f} dBFS")
show(rd, r, v, CAPTURE.name); plt.show()

## Task (a): the chirp count sets the velocity cell

The Phaser app's presets all send 256 chirps, so this cell makes the shorter bursts: it keeps only the first `M` chirps of your burst, makes the map, and reads, in the hand's range cell from the map above, the hand's velocity, its level, the width of its peak 3 dB down in velocity, and the noise floor.

1. Run it as it stands (`M = 256`) and record the row in Table F.
2. **Change one line:** set `M = 128`, run it, and record the row. Then `M = 64`.

**Check:** the velocity cell, printed as `dv`, is $\lambda/(2M\ T_\text{PRI})$: 0.118, 0.237, and 0.473 m/s. On the recorded hand the width 3 dB down prints 0.15, 0.25, and 0.48 m/s: one to one and a half cells at every $M$, so it roughly doubles each time you halve $M$. The level stays within about half a decibel (−26.6, −26.5, −26.5 dBFS), because `radar_lab` divides each FFT by the sum of its window, while the noise floor rises 3 dB each time (−109.8, −106.8, −104.1 dBFS): half the chirps, half the integration gain of Lesson 27.

In [ ]:
M = 256                  # TASK (a): change to 128, then 64

short = {**cap, "iq": cap["iq"][:M]}
rd_m, r_m, v_m = rl.range_doppler(short)
dv = v_m[1] - v_m[0]
i = i_hand                                    # the hand's range cell, from the full burst's map
j = int(np.argmax(np.where(np.abs(v_m) > 0.3, rd_m[i], -400)))
a_level = rd_m[i, j]
a_width = width_3db(rd_m[i], j, dv)
a_floor = floor(rd_m, r_m, v_m)
print(f"M = {M}: dv = {dv:.3f} m/s (lambda/(2 M PRI) = {lam/(2*M*cap['PRI']):.3f}); "
      f"hand {peak_velocity(rd_m[i], j, v_m):+.2f} m/s at {a_level:.1f} dBFS, {a_width:.2f} m/s wide 3 dB down; "
      f"floor {a_floor:.1f} dBFS")

plt.figure(figsize=(7, 3))
plt.plot(v_m, rd_m[i], ".-")
plt.xlim(-5, 5); plt.ylim(-125, -10)
plt.xlabel("velocity (m/s), + closing"); plt.ylabel("power (dBFS)")
plt.title(f"the hand's range cell, {r_m[i]:.2f} m, M = {M}"); plt.grid(alpha=0.3); plt.show()

## Task (b): 2-pulse against 3-pulse

The cell runs the burst through `rl.mti`, the same cancellers as the app's **MTI Mode**, and compares the map with the one that has no canceller. It reads the wall's cell, the hand, and the noise floor in two places: in the hand's own velocity column, and near 14 m/s, the edge of the velocity window.

1. **Change one line, step 1:** replace `None` in `H_DB = None` with the canceller's predicted power gain at the hand's velocity, in dB, from `f_D`, `cap["PRI"]`, and `STAGES`, the number of 2-pulse stages in `KIND` (the gain of the lesson's canceller-response frame). Run the cell with `KIND = "2pulse"` and record the row.
2. **Change one line, step 2:** set `KIND = "3pulse"` and run it again.

**Check:** the hand drops by your `H_DB`, to within about half a decibel: on the recorded hand −13.8 dB for the 2-pulse canceller and −27.7 dB for the 3-pulse, against −13.7 and −27.4 dB at exactly 1.0 m/s. The wall falls into the floor; in the recorded data that is more than 100 dB, because the simulated chirps are timed perfectly, and on your kit it is however far chirp sync and the room allow. The floor does not change by the same amount everywhere: the noise passes through the same filter as the targets, so near 14 m/s it rises about 6 and 12 dB, and in the hand's own column it falls with the hand. Part 3 asks what that does to the hand's SNR.

In [ ]:
KIND = "2pulse"          # TASK (b), step 2: change to "3pulse"
STAGES = {"none": 0, "2pulse": 1, "3pulse": 2}[KIND]

rd0, r0, v0 = rl.range_doppler(cap)
rd1, r1, v1 = rl.range_doppler(rl.mti(cap, KIND))
i, j, h0 = mover(rd0, r0, v0)
v_hand = peak_velocity(rd0[i], j, v0)
h1 = rd1[i, np.argmin(np.abs(v1 - v0[j]))]
f_D = 2 * v_hand / lam                       # the hand's Doppler frequency, Hz

H_DB = None              # TASK (b), step 1: predicted power gain at f_D, in dB, for STAGES stages

iw, jw, w0 = strongest(rd0, r0, v0, 1.5, 6.0, -0.01, 0.01)
w1 = rd1[iw, np.argmin(np.abs(v1))]
col = lambda rd, v, vv: floor(rd, r0, v, vv - 0.15, vv + 0.15)
b_hand_change = h1 - h0
b_wall_drop = w0 - w1
print(f"{KIND}: hand at {v_hand:+.2f} m/s, f_D = {f_D:.1f} Hz: {h0:.1f} -> {h1:.1f} dBFS, "
      f"change {b_hand_change:+.1f} dB; predicted {H_DB if H_DB is None else round(H_DB, 1)}")
print(f"wall at {r0[iw]:.2f} m: {w0:.1f} -> {w1:.1f} dBFS, down {b_wall_drop:.0f} dB")
print(f"floor in the hand's column: {col(rd0, v0, v_hand):.1f} -> {col(rd1, v1, v_hand):.1f} dBFS; "
      f"near 14 m/s: {floor(rd0, r0, v0, 13.5, 14.5):.1f} -> {floor(rd1, r0, v1, 13.5, 14.5):.1f} dBFS")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
show(rd0, r0, v0, "no MTI", ax=ax[0]); show(rd1, r0, v1, KIND, ax=ax[1])
plt.tight_layout(); plt.show()

## Task (c): the spinner's lines and its spectrogram

The map shows the fidget spinner as a band of velocities at one range, made of lines. The first cell below makes the spinner's map and reads the band's range, its lines, and their spacing, the numbers Table C asks for. A spectrogram then cuts one range cell's slow-time signal into short pieces and takes the FFT of each, so it shows how that band changes during the burst. `rl.spectrogram` subtracts the cell's mean first, which removes the hub and the hand that holds it.

1. Run the first cell below to see the spinner's map and its lines. **Change one line:** set `R_SPIN` in the second cell to the range of the band, read from the map, and run it with `N_WIN = 64`. Record the row in Table F.
2. **Change one line:** set `N_WIN = 32` and run it again. Record the row.

**Check:** the recorded spinner sits at 0.47 m. Its lines print at ±0.68, ±1.36, ±2.05, and ±2.73 m/s, 0.68 m/s apart on average, for a spin rate of 15.0 rev/s; the outermost, 2.73 m/s, is the band's edge, just inside the $2\pi(0.03\ \text{m})(15\ \text{rev/s}) = 2.83$ m/s tip speed. In the spectrogram the band within 10 dB of its peak reaches ±2.84 m/s at `N_WIN = 64` and ±2.84 m/s at 32, the tip speed to within a Doppler cell either way. Halving `N_WIN` doubles the Doppler cell, from 0.47 to 0.95 m/s, and halves the time between columns, from 8 to 4 ms: the trade-off Lesson 31 starts from. For Table C's face-on row without a kit, set `SPIN` to `L29_spinner_faceon.npz` and run the first cell again.

In [ ]:
SPIN = DATA / "L29_spinner.npz"   # change to your own spinner burst (Procedure C)
spin = rl.load_capture(SPIN)
rd_s, r_s, v_s = rl.range_doppler(spin)
i_s, j_s, _ = mover(rd_s, r_s, v_s, r_hi=2.0)
spin_lines = lines(rd_s[i_s], v_s)
pos = np.sort(spin_lines[spin_lines > 0])
spin_spacing = np.mean(np.diff(np.r_[0.0, pos])) if len(pos) else np.nan   # the lines sit at multiples of it
spin_edge = np.abs(spin_lines).max()
print(f"band at {r_s[i_s]:.2f} m; lines (m/s): {', '.join(f'{x:+.2f}' for x in np.sort(spin_lines))}")
print(f"edge {spin_edge:.2f} m/s; line spacing {spin_spacing:.2f} m/s, "
      f"so {2 * spin_spacing / (3 * lam):.1f} rev/s for three arms")
show(rd_s, r_s, v_s, SPIN.name, r_max=2.0); plt.show()

In [ ]:
R_SPIN = None            # TASK (c), step 1: the spinner's range from the map, in m
N_WIN = 64               # TASK (c), step 2: change to 32
HOP = N_WIN // 4

if R_SPIN is None:
    print("Set R_SPIN to the range of the spinner's band on the map above, then run this cell again.")
else:
    k = int(np.argmin(np.abs(rl.range_axis(spin) - R_SPIN)))     # the range FFT bin, as spectrogram wants it
    t, f, S = rl.spectrogram(spin, k, n_win=N_WIN, hop=HOP)
    vel = f * lam / 2
    mean_db = 10 * np.log10(np.mean(10 ** (S / 10), axis=1))
    band = vel[mean_db - mean_db.max() > -10]
    c_cell, c_step = vel[1] - vel[0], (t[1] - t[0]) * 1e3
    c_band = (band.min(), band.max())
    print(f"bin {k}, {rl.range_axis(spin)[k]:.2f} m: Doppler cell {c_cell:.2f} m/s, "
          f"columns {c_step:.0f} ms apart; band within 10 dB of its peak: "
          f"{c_band[0]:+.2f} to {c_band[1]:+.2f} m/s")
    plt.figure(figsize=(7, 3.5))
    plt.pcolormesh(t * 1e3, vel, S, vmin=-40, vmax=0, cmap="Blues", shading="auto")
    plt.xlabel("time (ms)"); plt.ylabel("velocity (m/s), + closing")
    plt.title(f"{SPIN.name}, {rl.range_axis(spin)[k]:.2f} m, N_WIN = {N_WIN}")
    plt.colorbar(label="dB"); plt.show()

## Any burst through the cancellers

No task here. This cell reads one range cell of any burst with each canceller: the strongest cell within half a range cell of `R_READ` and within 2 m/s of zero, its velocity, and its level. Use it for Table E's rows without a kit, `L29_hand_still.npz` and `L29_hand_across.npz` at 0.8 m, or to check a reading of your own.

**Check:** the recorded hand held still drops 58 dB through the 2-pulse canceller and about 100 (into the floor) dB through the 3-pulse; the hand moved across the beam, 33 and 66 dB. Both drop far more than the pushing hand's 14 and 27 dB.

In [ ]:
READ = DATA / "L29_hand_still.npz"   # any burst: L29_hand_across.npz, your own, ...
R_READ = 0.8                         # the range to read, m

burst = rl.load_capture(READ)
read_db = {}
for kind in ("none", "2pulse", "3pulse"):
    rd_k, r_k, v_k = rl.range_doppler(rl.mti(burst, kind))
    near = np.abs(r_k - R_READ) <= (r_k[1] - r_k[0]) / 2
    ik, jk, lk = strongest(rd_k, r_k, v_k, r_k[near].min(), r_k[near].max(), -2.0, 2.0)
    read_db[kind] = lk
    print(f"{READ.name}, {kind:>6}: strongest cell {r_k[ik]:.2f} m, {v_k[jk]:+.2f} m/s, {lk:.1f} dBFS"
          + ("" if kind == "none" else f", {lk - read_db['none']:+.1f} dB"))

## Before you close

Save the notebook with your changes and your plots, and upload the `.ipynb` with the packet. Keep it, `radar_lab.py`, and your bursts in the Module 4 folder: the L30 Lab runs CFAR on range-Doppler maps of this burst, with and without MTI. Anything you do beyond these three tasks, such as another capture, another range cell, or the 3-pulse canceller on the spinner, belongs in your report's Extra Effort.